In [317]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

In [318]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
k, n = 3, 4
#M = matroids.CompleteGraphic(n)
M = matroids.Uniform(k, n)
#M = matroids.catalog.Fano()
L = M.lattice_of_flats()
r = M.rank()
rank_func = lambda x : M.rank(x)

#e = next(x for x in M.groundset() if x  not in M.coloops())
e = list(M.groundset())[0]
print(M.groundset())
print("deleting ", e)
mdele = M.delete(e)
Eq_M = create_deformed_algebra(M, base_ring=P)
Eq_mdele = create_deformed_algebra(mdele, base_ring=P)
E_mdele = Eq_mdele.natural()

E = Eq_M.natural()
E_basis = E.basis()
E_mdele_basis = E_mdele.basis()

C = Eq_M.C()
C_dele = Eq_mdele.C()
C_basis = C.basis()
C_dele_basis = C_dele.basis()

flats_keys = list(E_basis.keys())

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())
mu_rev = lambda M: P(x**M.rank() * chibar(M)(0))
mu = lambda M: P(chibar(M)(0))

print(chibar(M))
print(mu_rev(M))
print(kl_poly(M))

frozenset({0, 1, 2, 3})
deleting  0
x^2 - 3*x + 3
3*x^3
1 + 2*x


In [319]:
def flat_division(M, e):
    L = M.lattice_of_flats()
    flat_div = {'S':set(), 'T':set(), 'A':set(), 'B':set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div

flat_div = flat_division(M, e)
#flat_div['S'].remove(frozenset())

def convolve(f, g, M):
    E = M.groundset()
    flats = list(M.lattice_of_flats())
    sum = P(0)
    for F in flats:
        sum += f(M.delete(E - F))(x) * g(M.contract(F))(x)
    return sum

conv = lambda f, g: convolve(f, g, M)

def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()

rightAugpoly = lambda M: convolve(chow, rev(kl_poly), M)
zeta = lambda M: P(1)
klrev = rev(kl_poly)

def leftAugpoly(M):
    return convolve(grev, chow_polynomial, M)

In [320]:
print("rank of M: ", r)
print("chow polynomial of M: ", chow_polynomial(M))
print(reduced_characteristic_polynomial(M))
for F in flat_div['S']:
    print(F)

rank of M:  3
chow polynomial of M:  x^2 + 7*x + 1
x^2 - 3*x + 3
frozenset()
frozenset({2})
frozenset({1})
frozenset({3})


In [321]:
p1 = convolve(mu_rev, chow_polynomial, M)
p2 = convolve(chow_polynomial, mu_rev, M)


print(p1)
print(p2)
print((p1-p2))


-1 - 3*x - 3*x^2 + 3*x^3
-1 - x - 3*x^2 + 3*x^3
-2*x


In [322]:


def deleconj(M, e, f):
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    sum2 = 0
    for F in flat_div['S']:
        poly1 = chow_polynomial(M.contract(F))
        poly2 = klrev(M.delete(M.groundset() - F))
        term = x * poly1 * poly2
        sum2 += term
        print(F)
        print("--------------------------------")
        print("poly1: ", poly1)
        print("poly2: ", poly2)
        print("term: ", term)
        print("--------------------------------")
        print("********************************")
    print("summation equals:", sum2)
    print("f(M):     ", og_matroid_poly)
    print("f(M\\e):  ", deleted_matroid_poly)
    print("difference: ", og_matroid_poly - deleted_matroid_poly)
    return P(deleted_matroid_poly + sum2)

def deleconjp2(M, e, f):
    sum2 = 0
    for F in flat_div['T']:
        poly1 = chow_polynomial(M.contract(F))
        poly2 = f(M.delete(M.groundset() - F -{e}))
        term = poly1 * poly2
        sum2 += term
    return P(sum2)

right = deleconj(M, e, rightAugpoly)
print("right", deleconj(M, e, rightAugpoly))
print("difference with right ", rightAugpoly(M) - right)

frozenset()
--------------------------------
poly1:  x^2 + 7*x + 1
poly2:  1
term:  x + 7*x^2 + x^3
--------------------------------
********************************
frozenset({2})
--------------------------------
poly1:  x + 1
poly2:  x
term:  x^2 + x^3
--------------------------------
********************************
frozenset({1})
--------------------------------
poly1:  x + 1
poly2:  x
term:  x^2 + x^3
--------------------------------
********************************
frozenset({3})
--------------------------------
poly1:  x + 1
poly2:  x
term:  x^2 + x^3
--------------------------------
********************************
summation equals: x + 10*x^2 + 4*x^3
f(M):      1 + 13*x + 13*x^2 + x^3
f(M\e):   1 + 7*x + 7*x^2 + x^3
difference:  6*x + 6*x^2
frozenset()
--------------------------------
poly1:  x^2 + 7*x + 1
poly2:  1
term:  x + 7*x^2 + x^3
--------------------------------
********************************
frozenset({2})
--------------------------------
poly1:  x + 1
poly2:  x
te

In [323]:
print(deleconjp2(M, e, rightAugpoly))

4 + 11*x + 4*x^2


In [324]:
print(rightAugpoly(M))
print(leftAugpoly(M))
print(rightAugpoly(M.contract(e))(x^2)*(x+x^-1))

1 + 13*x + 13*x^2 + x^3
1 + 11*x + 11*x^2 + x^3
x^-1 + 5*x + 5*x^3 + x^5
